In [ ]:
import requests

base_url = "http://192.168.157.163:8009"

### 下載模型到本地暫存區

In [ ]:
# model_name = "google/gemma-3-270m-it"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

data = {
    "model_source": "huggingface",
    "model_name": model_name
}

url = f"{base_url}/models/download"
response = requests.post(url, json=data)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    print(result["message"])
    print("模型暫存區:", result["details"])
    model_path = result["details"]
else:
    print(f"錯誤: {response.status_code} - {response.text}")
    


模型下載請求已處理
模型暫存區: /app/tmp/models/google_gemma-3-270m-it


### 下載資料集到本地暫存區

In [3]:
# dataset_name = "rajpurkar/squad_v2"
dataset_name = "yentinglin/TaiwanChat"
# dataset_name = "kigner/ruozhiba-llama3-tt"

data = {
  "dataset_name": dataset_name,
  "dataset_source": "huggingface",
  "extract_multimedia": False
}

# url = f"{base_url}/datasets/download_from_network"
# response = requests.post(url, json=data)

# # 檢查回應狀態
# if response.status_code == 200:
#     result = response.json()
#     print(result["message"])
#     print("資料集暫存區:", result["local_path"])
#     dataset_path = result["local_path"]
# else:
#     print(f"錯誤: {response.status_code} - {response.text}")

dataset_path = "/app/tmp/datasets/yentinglin_TaiwanChat"

### 提交訓練任務

In [ ]:
dataset_config = {
	"dataset_name_or_path": dataset_name,
	"cache_dir": dataset_path,
	"max_length": 2048,
	"train_size": 1000, # 訓練資料筆數
	"val_size": 500 # 驗證資料筆數
}

experiment_name = "_".join(
    [
        model_name.split('/')[1].split('-')[0],
        dataset_name.split('/')[1], 
        "finetune"
    ]
)

training_config = {
    "experiment_name": experiment_name,
    "model_name_or_path": model_path,
    "max_epochs": 10,
	"batch_size": 1,
	"gradient_accumulation_steps": 4,
	"learning_rate": 0.00002,
	"logging_steps": 5,
	"lora_config": {
		"bias": "none",
		"lora_alpha": 16,
		"lora_dropout": 0.05,
		"r": 8,
		"target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"]
	},
	"use_bfloat16": True,
	"use_flash_attn": False,
	"val_check_interval": 1.0,
	"warmup_steps": 100,
	"weight_decay": 0.01
}

submission = {
    "training_config": training_config,
    "dataset_config": dataset_config,
    "task_type": "instruction",
    "select_multiple_gpus": False,
    "vram_budget_gb": 15
}

url = f"{base_url}/training/submit"
response = requests.post(url, json=submission)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    print("任務 ID:", result["job_id"])
    print("狀態:", result["status"])
    print("使用的GPU:", result["gpu_id"])
    job_id = result["job_id"]
else:
    print(f"錯誤: {response.status_code} - {response.text}")

任務 ID: 6aab4db5014d4e
狀態: running
使用的GPU: [0]


### 追蹤任務進度

In [7]:
url = f"{base_url}/training/status/{job_id}"
response = requests.get(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    print("狀態", result["status"])
    display(result["metrics"])
else:
    print(f"錯誤: {response.status_code} - {response.text}")


錯誤: 404 - {"detail":"Job ID '1e4981ca8a8a41' not found."}


### 列出所有訓練任務

In [83]:
url = f"{base_url}/training/list"
response = requests.get(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    display(result)
else:
    print(f"錯誤: {response.status_code} - {response.text}")

[{'job_id': 'ae506324e6ed45',
  'gpu_id': [0],
  'status': 'running',
  'config': {'task_type': 'instruction',
   'vram_budget_gb': 10.0,
   'select_multiple_gpus': False,
   'trainer_config': {'model_name_or_path': '/app/tmp/models/google_gemma-3-270m-it',
    'use_bfloat16': True,
    'use_flash_attn': False,
    'weight_decay': 0.01,
    'warmup_ratio': None,
    'lora_config': {'bias': 'none',
     'lora_alpha': 16,
     'lora_dropout': 0.05,
     'r': 8,
     'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj']},
    'max_epochs': 10,
    'batch_size': 1,
    'learning_rate': 2e-05,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 100,
    'logging_steps': 5,
    'val_check_interval': 0.5,
    'checkpoint_dir': './checkpoints',
    'log_dir': './logs',
    'accelerator': 'auto',
    'devices': 1,
    'strategy': 'auto',
    'precision': '16-mixed',
    'experiment_name': 'gemma_TaiwanChat_finetune',
    'run_name': 'ae506324e6ed45',
    'gradient_checkpointing': Fal

### 取消訓練任務

In [36]:
url = f"{base_url}/training/cancel/{job_id}"
response = requests.post(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    display(result)
else:
    print(f"錯誤: {response.status_code} - {response.text}")


{'message': 'Cancellation request processed for Job ID: 7fa59a31a7c143'}

### 刪除任務紀錄

In [109]:
url = f"{base_url}/training/delete/{job_id}"
response = requests.delete(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    display(result)
else:
    print(f"錯誤: {response.status_code} - {response.text}")

{'message': "Job ID '2b58f2021c874b' successfully deleted."}